In [1]:
!pip install sentencepiece
!pip install nltk rouge-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.9 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.2/800.2 kB 64.2 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=d9e3a3ade049321f96acadea39bd99028436bb7b5bd89213cd4ada440470f4de
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge-score

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
import os
import shutil
import sys
import types
import transformers.pytorch_utils

os.environ["HF_HOME"] = "/workspace/huggingface_cache"


## Retrieval-Augmented Spec Truncation (RAST)

In [ ]:
import numpy as np

def smart_downsample(raw_data, claim_text, max_points=15):
    if len(raw_data) <= max_points:
        return raw_data

    n = len(raw_data)
    claim_text_lower = claim_text.lower()

    def get_y(pt):
        if isinstance(pt, (list, tuple)):
            if len(pt) >= 2: return float(pt[1])
            if len(pt) == 1: return float(pt[0])
            raise ValueError("Empty point")
        return float(pt)

    def get_x_str(pt):
        if isinstance(pt, (list, tuple)) and len(pt) >= 1:
            return str(pt[0]).strip().lower()
        return ""

    # Extract numeric y-values where possible
    y_vals = {}
    for i, pt in enumerate(raw_data):
        try:
            y_vals[i] = get_y(pt)
        except (ValueError, TypeError):
            pass

    keep_indices = {0, n - 1}  # first and last always

    # Global max and min — guaranteed
    if y_vals:
        keep_indices.add(max(y_vals, key=y_vals.get))
        keep_indices.add(min(y_vals, key=y_vals.get))

    # Local extrema ranked by prominence (deviation from neighbors)
    ranked_extrema = []
    for i in range(1, n - 1):
        if i not in y_vals or (i-1) not in y_vals or (i+1) not in y_vals:
            continue
        yc, yp, yn = y_vals[i], y_vals[i-1], y_vals[i+1]
        if (yc > yp and yc > yn) or (yc < yp and yc < yn):
            prominence = abs(yc - (yp + yn) / 2)
            ranked_extrema.append((prominence, i))
    ranked_extrema.sort(reverse=True)

    for _, idx in ranked_extrema:
        if len(keep_indices) >= max_points:
            break
        keep_indices.add(idx)

    # Claim keyword matching — word-boundary safe, min 3 chars
    if len(keep_indices) < max_points:
        import re
        for i, pt in enumerate(raw_data):
            if i in keep_indices or len(keep_indices) >= max_points:
                continue
            x_str = get_x_str(pt)
            if len(x_str) >= 3:
                # Word boundary match to avoid "Jan" inside "January"
                if re.search(r'\b' + re.escape(x_str) + r'\b', claim_text_lower):
                    keep_indices.add(i)

    # Fill remainder with uniform stride
    if len(keep_indices) < max_points:
        remaining_budget = max_points - len(keep_indices)
        available = [i for i in range(n) if i not in keep_indices]
        if available and remaining_budget > 0:
            stride = max(1.0, len(available) / remaining_budget)
            for j in range(remaining_budget):
                idx_to_add = available[min(int(j * stride), len(available) - 1)]
                keep_indices.add(idx_to_add)

    return [raw_data[i] for i in sorted(keep_indices)[:max_points]]

In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading Semantic Retriever...")
# all-MiniLM-L6-v2 is only 22M parameters and extremely fast
semantic_retriever = SentenceTransformer("all-MiniLM-L6-v2")


def verbalize_series_for_retrieval(ser: dict) -> str:
    """Convert series dict to a natural language summary for semantic matching."""
    parts = []
    name = ser.get("name", "unknown series")
    parts.append(f"Series named {name}.")

    stats = ser.get("stats", {})
    if stats.get("max_value") is not None:
        parts.append(f"Maximum value {stats['max_value']} at {stats.get('max_at_the_ref_point', 'unknown')}.")
    if stats.get("min_value") is not None:
        parts.append(f"Minimum value {stats['min_value']} at {stats.get('min_at_the_ref_point', 'unknown')}.")

    trend = ser.get("trend", {})
    if trend.get("global_trend"):
        parts.append(f"Overall trend is {trend['global_trend'].replace('_', ' ')}.")

    data = ser.get("data", [])
    if data and isinstance(data[0], (list, tuple)) and len(data[0]) >= 2:
        try:
            first = data[0]
            last = data[-1]
            parts.append(f"Starts at {first[0]}: {first[1]}, ends at {last[0]}: {last[1]}.")
        except (IndexError, TypeError):
            pass

    return " ".join(parts)


def retrieve_top_k_series_semantic(claim, spec_json_str, k=None):
    """
    Keeps the top-k most claim-relevant series using semantic similarity
    on verbalized series summaries. Always protects reference/threshold series.
    Dynamically sets k based on claim signals if k is None.
    """
    try:
        spec = json.loads(spec_json_str)
    except Exception:
        return spec_json_str

    series_list = spec.get('ser', [])
    if not isinstance(series_list, list):
        return spec_json_str

    # Separate protected series (reference lines, thresholds)
    protected = []
    candidates = []
    for s in series_list:
        name = s.get("name", "").lower()
        if any(kw in name for kw in ("threshold", "reference", "target",
                                      "average", "mean", "baseline")):
            protected.append(s)
        else:
            candidates.append(s)

    # Dynamic k: comparative claims need more series
    if k is None:
        claim_lower = claim.lower()
        comparative_signals = ["higher", "lower", "greater", "less", "more than",
                                "compared", "versus", "between", "rank", "largest",
                                "smallest", "all", "both", "each"]
        is_comparative = any(s in claim_lower for s in comparative_signals)
        k = min(len(candidates), 5 if is_comparative else 3)

    # If already within budget, return as-is
    if len(candidates) <= k:
        return spec_json_str

    # Verbalize candidates for better semantic matching
    candidate_texts = [verbalize_series_for_retrieval(s) for s in candidates]
    docs = [claim] + candidate_texts

    with torch.no_grad():
        embs = semantic_retriever.encode(docs, convert_to_tensor=True,
                                          normalize_embeddings=True)
        sims = torch.nn.functional.cosine_similarity(embs[0:1], embs[1:])

    top_k_indices = sims.topk(k).indices.sort().values.tolist()
    selected_candidates = [candidates[i] for i in top_k_indices]

    # Reconstruct: protected first (preserve original order), then selected
    original_order = {id(s): i for i, s in enumerate(series_list)}
    final_series = sorted(
        protected + selected_candidates,
        key=lambda s: original_order.get(id(s), 999)
    )

    spec['ser'] = final_series
    if len(final_series) < len(series_list):
        spec['is_truncated'] = True

    return json.dumps(spec, separators=(',', ':'))

In [ ]:
import json

def prune_spec_for_encoder(raw_spec_dict, claim_text):
    """
    Minifies the JSON string, intelligently downsamples massive data arrays, 
    and uses a recursive global sweeper to kill VLM autoregressive loops ANYWHERE in the JSON.
    """
    if not isinstance(raw_spec_dict, dict):
        return "{}"
        
    cleaned_spec = raw_spec_dict.copy()
    
    # --- [FIX 4] THE GHOST SPEC GATEKEEPER ---
    # If the chart has no series, it is mathematically dead. 
    # Reject it immediately so the description field doesn't mask it.
    if 'ser' not in cleaned_spec or not isinstance(cleaned_spec['ser'], list) or len(cleaned_spec['ser']) == 0:
        return "{}"
        
    # --- HELPER 1: Safe Float Compression ---
    def compress_val(v):
        """Rounds floats to 2 decimals, converts .0 floats to ints, leaves strings alone."""
        if isinstance(v, float):
            return int(v) if v.is_integer() else round(v, 2)
        return v

    # --- HELPER 2: The Universal Loop Killer ---
    def kill_repeating_loops(lst, max_consecutive=2, max_length=20):
        """Removes consecutive duplicates and hard-caps length at 20."""
        if not isinstance(lst, list): return lst
        cleaned_list = []
        last_item = None
        consecutive_count = 0
        
        for item in lst:
            if item == last_item:
                consecutive_count += 1
            else:
                last_item = item
                consecutive_count = 1
                
            if consecutive_count <= max_consecutive:
                cleaned_list.append(compress_val(item))
                
        return cleaned_list[:max_length]

    # --- HELPER 3: Global List Sweeper (Updated for Lists of Dicts) ---
    def sweep_for_loops(obj):
        """Recursively hunts down any list in the JSON and applies the Loop Killer."""
        if isinstance(obj, dict):
            for k, v in list(obj.items()):
                if k == 'data': 
                    continue # Leave raw data arrays to the explicit downsampler below
                if isinstance(v, list):
                    # If it's a list of primitives (strings/numbers), kill loops
                    if all(isinstance(x, (str, int, float, bool)) for x in v):
                        obj[k] = kill_repeating_loops(v)
                    # If it's a list of dicts, recurse deeper into them
                    else:
                        for item in v:
                            sweep_for_loops(item)
                elif isinstance(v, dict):
                    sweep_for_loops(v)

    # 1. Ensure structural conformity & nuke bloated relation arrays
    cleaned_spec.pop('legend', None)
    
    topo_dict = cleaned_spec.get('topo', {})
    raw_topo_type = topo_dict.get('type', '') if isinstance(topo_dict, dict) else ''
    topo_type = raw_topo_type.lower() if isinstance(raw_topo_type, str) else ''
    
    # 2. Globally sweep and kill loops in ALL fields
    sweep_for_loops(cleaned_spec)
    
    spec_json = json.dumps(cleaned_spec, separators=(',', ':'))
    spec_json = retrieve_top_k_series_semantic(claim_text, spec_json)
    cleaned_spec = json.loads(spec_json)
    series_list = cleaned_spec.get('ser', [])
    
    # 3. Explicitly Sub-sample Series Data safely 
    # Step 2: Per-series pruning AFTER retrieval
    for ser in series_list:
        if isinstance(ser, dict):
            ser.pop('ds', None)
            ser.pop('is_subsampled', None)
            ser.pop('critical_points_retained', None)
            # Keep y_ref until after retrieval — pop it here instead
            ser.pop('y_ref', None)

            if 'pie' in topo_type:
                ser.pop('trend', None)
                ser.pop('stats', None)

            if 'data' in ser and isinstance(ser['data'], list):
                ser['data'] = smart_downsample(ser['data'], claim_text, max_points=15)
                ser['data'] = [[compress_val(v) for v in pt] 
                                if isinstance(pt, list) else compress_val(pt)
                                for pt in ser['data']]

    return json.dumps(cleaned_spec, separators=(',', ':'))

In [ ]:
def spec_to_prose(spec_json_str: str) -> str:
    """
    Converts a pruned ChartSpec JSON string into a natural-language premise
    for the ModernBERT cross-encoder.

    Data formats per chart type (from SpecExtraction.py):
      - Line / Bar / Scatter:  [x_val, y_val]
      - Pie:                   [percentage]          (single-element list)
      - Area (un/stacked):     [x_val, y_max, y_min]
      - Box plot:              [category, min, q1, median, q3, max]
      - Histogram:             [bin_start, bin_end, height]
      - Bubble scatter:        [x_val, y_val, size]
      - Colormap scatter:      [x_val, y_val, color_scalar]
      - Threshold (axhline):   [x_start, y_val] / [x_end, y_val]  (2-point flat line)
      - Hybrid / outlier:      treated as scatter series with appropriate name

    Special series name conventions (from SpecExtraction.py):
      - "Box Distribution"         → box plot distribution series
      - "scatter_series_outliers"  → outlier points extracted from box plots
      - starts with "threshold_"   → reference / threshold line
      - starts with "scatter_"     → scatter / outlier overlay on another chart type
      - starts with "area_layer_"  → area band layer
    """
    try:
        spec = json.loads(spec_json_str)
    except Exception:
        return spec_json_str if isinstance(spec_json_str, str) else ""

    if not isinstance(spec, dict):
        return str(spec_json_str)

    # ── Topology ──────────────────────────────────────────────────────────────
    topo  = spec.get("topo", {})
    ctype = topo.get("type", "unknown") if isinstance(topo, dict) else "unknown"
    csub  = topo.get("sub",  "")        if isinstance(topo, dict) else ""
    type_str = f"{csub} {ctype}".strip() if csub else ctype
    lines = [f"Chart type: {type_str}"]

    title = spec.get("title")
    if title:
        lines.append(f"Title: {title}")

    # ── Axes ──────────────────────────────────────────────────────────────────
    for ax in spec.get("axes", []):
        if not isinstance(ax, dict):
            continue
        name = ax.get("name", "?")
        lbl  = ax.get("lbl")
        dom  = ax.get("dom", [])
        scl  = ax.get("scl", "")

        ax_parts = [f"Axis '{name}'"]
        if lbl:
            ax_parts.append(f"label '{lbl}'")
        if scl and scl != "linear":
            ax_parts.append(f"scale {scl}")
        if isinstance(dom, list) and len(dom) == 2:
            ax_parts.append(f"range {dom[0]} to {dom[1]}")
        elif isinstance(dom, list) and len(dom) > 2:
            # categorical axis — list of category labels
            cats = ", ".join(str(d) for d in dom[:10])
            suffix = f" (+ {len(dom)-10} more)" if len(dom) > 10 else ""
            ax_parts.append(f"categories: {cats}{suffix}")
        lines.append(" ".join(ax_parts))

    # ── Series helpers ────────────────────────────────────────────────────────
    def _classify_series(name: str, chart_type: str) -> str:
        """Return a prose label describing what kind of series this is."""
        nl = name.lower()
        if nl.startswith("threshold_") or nl == "threshold":
            return "threshold_line"
        if nl == "scatter_series_outliers" or nl.startswith("scatter_series_outlier"):
            return "outlier_points"
        if nl.startswith("scatter_"):
            return "scatter_overlay"
        if nl == "box distribution":
            return "box_distribution"
        if nl.startswith("area_layer_"):
            return "area_layer"
        return "data_series"

    def _fmt_point(pt, series_class: str, chart_type: str) -> str:
        """Format a single data point according to its chart type / series class."""
        if not isinstance(pt, (list, tuple)):
            return str(pt)

        # Threshold lines: [x, y] where both points share the same y → just report y
        if series_class == "threshold_line":
            if len(pt) >= 2:
                return f"y={pt[1]}"
            return str(pt[0])

        # Box distribution: [category, min, q1, median, q3, max]
        if series_class == "box_distribution" and len(pt) >= 6:
            return (f"{pt[0]}("
                    f"min={pt[1]}, q1={pt[2]}, "
                    f"med={pt[3]}, q3={pt[4]}, max={pt[5]})")

        # Histogram: [bin_start, bin_end, height]
        if chart_type == "histogram" and len(pt) == 3:
            return f"[{pt[0]}-{pt[1]}]:{pt[2]}"

        # Pie: [percentage]  (single float, stored as fraction 0-1 or percent)
        if chart_type == "pie":
            if len(pt) == 1:
                val = pt[0]
                # SpecExtraction stores as fraction (angle/360)
                pct = round(float(val) * 100, 1) if float(val) <= 1.0 else round(float(val), 1)
                return f"{pct}%"
            return str(pt[0])

        # Area: [x_val, y_max, y_min]
        if chart_type in ("area", "stacked area") and len(pt) == 3:
            return f"({pt[0]}: {pt[1]}↑{pt[2]}↓)"

        # Bubble scatter: [x, y, size]
        if len(pt) == 3 and series_class in ("scatter_overlay", "data_series"):
            return f"({pt[0]}, {pt[1]}, size={pt[2]})"

        # Colormap scatter: [x, y, color_scalar]
        if len(pt) == 4:
            return f"({pt[0]}, {pt[1]}, size={pt[2]}, c={pt[3]})"

        # Default: [x, y]
        if len(pt) >= 2:
            return f"({pt[0]}: {pt[1]})"

        return str(pt[0])

    # ── Separate series by role ───────────────────────────────────────────────
    threshold_series = []
    outlier_series   = []
    scatter_overlays = []
    main_series      = []

    for ser in spec.get("ser", []):
        if isinstance(ser, str):
            try:    ser = json.loads(ser)
            except: continue
        if not isinstance(ser, dict):
            continue

        sc = _classify_series(ser.get("name", ""), ctype)
        if sc == "threshold_line":
            threshold_series.append((ser, sc))
        elif sc == "outlier_points":
            outlier_series.append((ser, sc))
        elif sc == "scatter_overlay":
            scatter_overlays.append((ser, sc))
        else:
            main_series.append((ser, sc))

    # ── Render main series ────────────────────────────────────────────────────
    def _render_series(ser: dict, sc: str) -> None:
        name  = ser.get("name", "unnamed")
        data  = ser.get("data",  [])
        stats = ser.get("stats", {})
        trend = ser.get("trend", {})
        color = ser.get("color")
        y_ref = ser.get("y_ref")          # kept until after retrieval

        # Header
        header_parts = [f"Series '{name}'"]
        if color:
            header_parts.append(f"color {color}")
        if y_ref and y_ref != "y_axis":
            header_parts.append(f"on {y_ref}")
        header = " ".join(header_parts)

        # Data points
        if isinstance(data, list) and data:
            pts = [_fmt_point(p, sc, ctype) for p in data[:15]]
            data_str = ", ".join(pts)
            suffix = f" … (+{len(data)-15} more)" if len(data) > 15 else ""
            line = f"{header}: {data_str}{suffix}"
        else:
            line = header

        # Stats — handles both line/area keys and bar/scatter keys
        if isinstance(stats, dict) and stats:
            stat_parts = []

            # Extrema (line, area, bar)
            mx = stats.get("max_value", stats.get("max_upper_bound"))
            mn = stats.get("min_value", stats.get("min_upper_bound"))
            mx_at = stats.get("max_at_the_ref_point")
            mn_at = stats.get("min_at_the_ref_point")
            if mx is not None:
                at_str = f" at {mx_at}" if mx_at is not None else ""
                stat_parts.append(f"max={mx}{at_str}")
            if mn is not None:
                at_str = f" at {mn_at}" if mn_at is not None else ""
                stat_parts.append(f"min={mn}{at_str}")

            # Histogram modal bin
            modal = stats.get("modal_bin")
            max_freq = stats.get("max_frequency")
            if modal:
                stat_parts.append(f"modal_bin={modal} freq={max_freq}")

            # Scatter cluster statistics
            cluster = stats.get("cluster_statistics") or \
                      (stats if "centroid" in stats else None)
            if cluster:
                c = cluster.get("centroid")
                sp = cluster.get("spread_std_dev")
                if c:
                    stat_parts.append(f"centroid=({c[0]}, {c[1]})")
                if sp:
                    stat_parts.append(f"spread=(σx={sp[0]}, σy={sp[1]})")

            # Box plot distribution summary
            dist = stats.get("distribution_summary") or ser.get("distribution_summary")
            if dist and isinstance(dist, dict):
                avg_iqr = dist.get("average_iqr")
                if avg_iqr is not None:
                    stat_parts.append(f"avg_iqr={avg_iqr}")

            # Crossing points
            crossings = stats.get("crossing_points", [])
            for cp in crossings[:3]:          # cap at 3 to avoid verbosity
                pair = cp.get("series_pair", [])
                ref  = cp.get("ref", "?")
                val  = cp.get("value", cp.get("hi", "?"))
                if pair:
                    stat_parts.append(
                        f"crosses '{pair[1]}' near {ref} (val≈{val})")

            if stat_parts:
                line += ". " + ", ".join(stat_parts)

        # Trend
        if isinstance(trend, dict) and trend:
            gt = trend.get("global_trend", "")
            roc = trend.get("rate_of_change", "")
            intervals = trend.get("intervals", [])
            trend_parts = []
            if gt:
                trend_parts.append(gt.replace("_", " "))
            if roc and roc != "stable":
                trend_parts.append(f"rate: {roc}")
            for iv in intervals[:2]:
                xr = iv.get("x_range", [])
                tr = iv.get("trend", "")
                if xr and tr:
                    trend_parts.append(f"{xr[0]}→{xr[1]}: {tr}")
            if trend_parts:
                line += ". Trend: " + ", ".join(trend_parts)

        lines.append(line)

    for ser, sc in main_series:
        _render_series(ser, sc)

    # ── Threshold lines ───────────────────────────────────────────────────────
    for ser, sc in threshold_series:
        name = ser.get("name", "threshold")
        data = ser.get("data", [])
        color = ser.get("color", "")
        # Extract the constant y-value from the first point
        y_val = None
        if data and isinstance(data[0], (list, tuple)) and len(data[0]) >= 2:
            y_val = data[0][1]
        color_str = f" (color {color})" if color else ""
        if y_val is not None:
            lines.append(f"Threshold line '{name}' at y={y_val}{color_str}")
        else:
            lines.append(f"Threshold line '{name}'{color_str}")

    # ── Scatter overlays (hybrid charts) ─────────────────────────────────────
    for ser, sc in scatter_overlays:
        _render_series(ser, sc)

    # ── Outlier points ────────────────────────────────────────────────────────
    for ser, sc in outlier_series:
        name = ser.get("name", "outliers")
        data = ser.get("data", [])
        if data:
            pts = [_fmt_point(p, sc, ctype) for p in data[:10]]
            lines.append(f"Outlier points '{name}': {', '.join(pts)}")

    # ── Comparative relations (rel[]) ─────────────────────────────────────────
    for rel in spec.get("rel", []):
        if not isinstance(rel, dict):
            continue
        # Point-wise comparison: {ref, relation, ranking}
        relation = rel.get("relation", "")
        ranking  = rel.get("ranking",  [])
        ref      = rel.get("ref", "")
        # Global descriptions: {relation_type, description, ranking/peak_magnitude_ranking}
        description = rel.get("description", "")
        peak_rank   = rel.get("peak_magnitude_ranking", [])

        if relation:
            lines.append(f"Comparison: {relation}")
        if description:
            lines.append(f"Comparison: {description}")
        if ranking and ref:
            lines.append(f"Ranking at '{ref}': {' > '.join(str(r) for r in ranking)}")
        if peak_rank:
            lines.append(f"Peak ranking: {' > '.join(str(r) for r in peak_rank)}")

    return ". ".join(lines) + "."

## Setup, Imports, and Data Loading

In [ ]:
import json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

MAX_CLAIM_LEN = 256
MAX_SPEC_LEN = 4096 # ModernBERT native context
BATCH_SIZE = 2       
EFFECTIVE_BATCH_SIZE = 16 
ACCUMULATION_STEPS = max(1, EFFECTIVE_BATCH_SIZE // BATCH_SIZE)
EPOCHS = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

def load_and_prep_data(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    cleaned_data = []
    for row in data:
        raw_spec = row.get("extended_spec", {})
        if isinstance(raw_spec, dict):
            raw_spec.pop("description", None) 
            
        claim_text = row.get("claim", "")
            
        # --- FIX A & B: Pass claim_text for smart downsampling, remove redundant retrieval ---
        pruned_spec_str = prune_spec_for_encoder(raw_spec, claim_text)
        if not pruned_spec_str:
            pruned_spec_str = "{}"
            
        # --- MISSING LINK FIXED: Convert the pruned JSON to Prose ---
        prose_spec = spec_to_prose(pruned_spec_str)
            
        raw_label = str(row.get("label", "")).strip().upper()
        if raw_label in ["SUPPORTED", "TRUE", "1"]:
            label = 1
        elif raw_label in ["REFUTED", "CONTRADICTED", "FALSE", "0"]:
            label = 0
        else:
            continue 
            
        cleaned_data.append({
            "claim": claim_text,
            "description": row.get("title_description", ""), 
            "spec_str": prose_spec, # The dataset now stores the PROSE version
            "chart_type": row.get("chart_type", "unknown"),
            "reasoning_type": row.get("reasoning_type", ["unknown"]), 
            "label": label
        })
    return cleaned_data

print("Loading compiled datasets...")
train_data = load_and_prep_data("train_w_spec.json")
val_data = load_and_prep_data("validation_w_spec.json")
test_data = load_and_prep_data("test_1_w_spec.json") 
test2_data = load_and_prep_data("test_2_w_spec.json") 

print(f"Train samples: {len(train_data)} | Val samples: {len(val_data)}")

Using device: cuda
Loading compiled datasets...
Train samples: 7607 | Val samples: 953 | Test samples: 939 | Test 2 samples: 981


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

unique_lengths = []
seen_specs = set()

print("⏳ Tokenizing unique charts...")
all_data = train_data + val_data + test_data + test2_data

# Filter for unique charts and calculate lengths
for item in all_data:
    spec = item["spec_str"]
    if spec not in seen_specs:
        seen_specs.add(spec)
        tokens = tokenizer.encode(spec) 
        unique_lengths.append(len(tokens))

print(f"\nTotal Unique Charts Evaluated: {len(unique_lengths)}")
print(f"Max Spec Tokens: {max(unique_lengths)}")
print(f"Average Spec Tokens: {sum(unique_lengths)/len(unique_lengths):.0f}\n")

# 1. Mathematical Breakdown
bins = [0, 512, 1024, 2048, 4096, 10000]
labels = ["< 512 tokens", "512 - 1024 tokens", "1024 - 2048 tokens", "2048 - 4096 tokens", "> 4096 tokens"]

hist, _ = np.histogram(unique_lengths, bins=bins)

print("📊 Token Length Distribution (Unique Charts):")
print("-" * 55)
for label, count in zip(labels, hist):
    percentage = (count / len(unique_lengths)) * 100
    print(f"{label:>20}: {count:>5} charts ({percentage:.1f}%)")

# 2. Visual Histogram
plt.figure(figsize=(10, 6))
plt.hist(unique_lengths, bins=50, color='#4C72B0', edgecolor='black')

plt.axvline(2048, color='red', linestyle='dashed', linewidth=2, label='DeBERTa Max Cutoff (1024)')

plt.title("Distribution of JSON Spec Token Lengths (Unique Charts Only)")
plt.xlabel("Number of Subword Tokens")
plt.ylabel("Number of Unique Charts")
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

unique_lengths = []
unique_lengths_claim_only = []
seen_specs = set()

print("⏳ Tokenizing unique charts...")
all_data = train_data + val_data + test_data + test2_data

# Filter for unique charts and calculate lengths
for item in all_data:
    claim_text = str(item["claim"]) + " " + str(item.get("description", ""))
    spec_text = str(item["spec_str"])
    enc = tokenizer(claim_text, spec_text, truncation=False)
    combined_lengths.append(len(enc["input_ids"]))

print(f"\nTotal Unique Charts Evaluated: {len(unique_lengths)}")
print(f"Max claim_description Tokens: {max(unique_lengths)}")
print(f"Average claim_description Tokens: {sum(unique_lengths)/len(unique_lengths):.0f}\n")

print(f"\nTotal Unique Charts Evaluated: {len(unique_lengths_claim_only)}")
print(f"Max claim only Tokens: {max(unique_lengths_claim_only)}")
print(f"Average claim only Tokens: {sum(unique_lengths_claim_only)/len(unique_lengths_claim_only):.0f}\n")

# 1. Mathematical Breakdown
bins = [0, 256, 512, 1024, 2048]
labels = ["< 256 tokens", "256-512" ,"512 - 1024 tokens", "1024 - 2048 tokens"]

hist, _ = np.histogram(unique_lengths, bins=bins)

print("📊 Token Length Distribution (Unique Charts):")
print("-" * 55)
for label, count in zip(labels, hist):
    percentage = (count / len(unique_lengths)) * 100
    print(f"{label:>20}: {count:>5} charts ({percentage:.1f}%)")

# 2. Visual Histogram
plt.figure(figsize=(10, 6))
plt.hist(unique_lengths, bins=50, color='#4C72B0', edgecolor='black')

plt.axvline(256, color='red', linestyle='dashed', linewidth=2, label='DeBERTa Max Cutoff (1024)')

plt.title("Distribution of claim+description Token Lengths (Unique Charts Only)")
plt.xlabel("Number of Subword Tokens")
plt.ylabel("Number of Unique Charts")
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import json
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load the raw dataset so we have access to the image paths
with open("train_w_spec.json", 'r', encoding='utf-8') as f:
    raw_train_data = json.load(f)

seen_images = set()
outlier_charts = []

print("🔍 Hunting for >2048 token outliers...")

for row in raw_train_data:
    img_path = row.get("local_image_path", "Unknown_Path")
    
    # Skip if we already checked this chart
    if img_path in seen_images:
        continue
    seen_images.add(img_path)
    
    # Apply your current pruner
    pruned_spec_str = prune_spec_for_encoder(row.get("extended_spec", {}), claim_text)
    prose = spec_to_prose(pruned_spec_str)
    tokens = tokenizer.encode(prose)

    length = len(tokens)
    
    if length > 2048:
        outlier_charts.append((img_path, length))

# Sort from largest to smallest
outlier_charts.sort(key=lambda x: x[1], reverse=True)

print(f"\n🚨 Found {len(outlier_charts)} extreme outlier charts:")
print("-" * 60)
for img, length in outlier_charts:
    # Print just the filename, ignoring the 'local_images/' folder path
    filename = img.split('/')[-1] if '/' in img else img
    print(f"[{length:^6} tokens] {filename}")

## The Single-Stream Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
from transformers import DataCollatorWithPadding

class SingleStreamChartDataset(Dataset):
    def __init__(self, data, tok, max_length=MAX_SPEC_LEN):
        self.data = data
        self.tokenizer = tok
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Combine claim and description as the "Premise"
        claim_text = str(item["claim"]) + " " + str(item.get("description", ""))
        
        # The spec_text is now the output of spec_to_prose
        spec_text = str(item["spec_str"])
        
        # Encode as a single cross-attention stream: [CLS] Claim [SEP] Spec Prose [SEP]
        encoding = self.tokenizer(
            text=claim_text,
            text_pair=spec_text,
            max_length=self.max_length,
            truncation='only_second', # Safely truncates the Spec, NEVER the claim
            return_tensors="pt"
        )
        
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(item["label"], dtype=torch.float),
            # "chart_type": item["chart_type"],
            # "reasoning_type": item["reasoning_type"]
        }

# Instantiate dataloaders
train_dataset = SingleStreamChartDataset(train_data, tokenizer)
val_dataset = SingleStreamChartDataset(val_data, tokenizer)
test_dataset = SingleStreamChartDataset(test_data, tokenizer)
test2_dataset = SingleStreamChartDataset(test2_data, tokenizer)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=data_collator)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)
test2_loader = DataLoader(test2_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)

## The ModernBERT Architecture

In [ ]:
import torch.nn as nn
from transformers import AutoModel

class ModernBERTVerifier(nn.Module):
    def __init__(self, model_name="answerdotai/ModernBERT-base"):
        super(ModernBERTVerifier, self).__init__()
        
        print(f"🔨 Loading Single-Stream Cross-Encoder: {model_name}")
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        
        # Final Classification Head attached to the [CLS] token
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, input_ids, attention_mask, output_attentions=False):
        outputs = self.encoder(
            input_ids=input_ids, 
            attention_mask=attention_mask,
            output_attentions=output_attentions 
        )
        
        # Extract the [CLS] token representation
        cls_output = outputs.last_hidden_state[:, 0, :]
        logits = self.mlp(cls_output)
        
        # Return attentions only if requested (for the explainability module)
        # attns = outputs.attentions[-1] if output_attentions and outputs.attentions else None
        # In ModernBERTVerifier.forward:
        attns = outputs.attentions if output_attentions and outputs.attentions else None
        
        return logits.squeeze(-1), attns

# Instantiate and cast to device
model = ModernBERTVerifier()
SAFE_DTYPE = torch.float16 if not torch.cuda.is_bf16_supported() else torch.bfloat16
model = model.to(device=DEVICE, dtype=SAFE_DTYPE)

## Optimizer & Evaluation Setup

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score

def create_optimizer_and_scheduler(model, train_loader, epochs, accumulation_steps):
    no_decay = ['bias', 'norm.weight', 'layernorm.weight', 'layer_norm.weight']
    
    optimizer_grouped_parameters = [
        {'params': [p for n, p in model.encoder.named_parameters() if not any(nd in n for nd in no_decay)],
         'weight_decay': 0.01, 'lr': 2e-5},
        {'params': [p for n, p in model.encoder.named_parameters() if any(nd in n for nd in no_decay)],
         'weight_decay': 0.0, 'lr': 2e-5},
         
        {'params': [p for n, p in model.mlp.named_parameters() if not any(nd in n for nd in no_decay)],
         'weight_decay': 0.01, 'lr': 5e-5},
        {'params': [p for n, p in model.mlp.named_parameters() if any(nd in n for nd in no_decay)],
         'weight_decay': 0.0, 'lr': 5e-5}
    ]

    optimizer = AdamW(optimizer_grouped_parameters)
    total_steps = (len(train_loader) // accumulation_steps) * epochs
    warmup_steps = int(0.1 * total_steps) 

    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    return optimizer, scheduler

def evaluate_single_stream(model, dataloader, criterion, device):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            logits, _ = model(input_ids, attention_mask)
            loss = criterion(logits.to(torch.float32), labels.to(torch.float32))
            total_loss += loss.item()
            
            preds = (torch.sigmoid(logits) >= 0.5).long().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    avg_loss = total_loss / len(dataloader)
    
    return avg_loss, acc, f1

## The Training Loop

In [ ]:
from tqdm import tqdm

criterion = nn.BCEWithLogitsLoss()
optimizer, scheduler = create_optimizer_and_scheduler(model, train_loader, EPOCHS, ACCUMULATION_STEPS)

print(f"🚀 Initializing Single-Stream Training on {DEVICE}")
best_val_f1 = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0.0
    optimizer.zero_grad()
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for step, batch in enumerate(progress_bar):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        
        logits, _ = model(input_ids, attention_mask)
        loss = criterion(logits, labels.to(logits.dtype))
        
        loss = loss / ACCUMULATION_STEPS
        loss.backward()
        
        total_train_loss += (loss.item() * ACCUMULATION_STEPS)
        progress_bar.set_postfix({'loss': f"{loss.item() * ACCUMULATION_STEPS:.4f}"})
        
        if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
    avg_train_loss = total_train_loss / len(train_loader)
    val_loss, val_acc, val_f1 = evaluate_single_stream(model, val_loader, criterion, DEVICE)
    
    print(f"\nEpoch {epoch+1} Summary:")
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"Val Accuracy: {val_acc * 100:.2f}% | Val F1 (Macro): {val_f1:.4f}")
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        checkpoint_path = "best_modernbert_verifier.pt"
        torch.save(model.state_dict(), checkpoint_path)
        print(f"💾 Validation F1 improved! Saved new best model to {checkpoint_path}")

In [ ]:
import pandas as pd
from tqdm import tqdm

# Load the best checkpoint from training
model.load_state_dict(torch.load("best_modernbert_verifier.pt", map_location=DEVICE))
model.eval()
print(f"✅ Loaded best checkpoint from 'best_modernbert_verifier.pt'")

def run_test_evaluation(model, dataloader, device):
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating Test Set"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].cpu().numpy()
            
            logits, _ = model(input_ids, attention_mask)
            preds = (torch.sigmoid(logits) >= 0.5).long().cpu().numpy()
            
            all_preds.extend(preds)
            all_labels.extend(labels)
            
    acc = accuracy_score(all_labels, all_preds) * 100
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0) * 100
    return acc, f1

t1_acc, t1_f1 = run_test_evaluation(model, test_loader, DEVICE)
t2_acc, t2_f1 = run_test_evaluation(model, test2_loader, DEVICE)
avg_acc = (t1_acc + t2_acc) / 2.0

print(f"\nTest 1  →  Acc: {t1_acc:.1f}%   Macro-F1: {t1_f1:.1f}")
print(f"Test 2  →  Acc: {t2_acc:.1f}%   Macro-F1: {t2_f1:.1f}")
print(f"Average →  Acc: {avg_acc:.1f}%")

## Attention Tracing & Flan-T5 Rationale Generation

In [ ]:
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from torch.utils.data import ConcatDataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import re

print("Loading Flan-T5 for rationale synthesis...")
tokenizer_t5 = AutoTokenizer.from_pretrained("google/flan-t5-base")
model_t5 = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base").to(DEVICE)
model_t5.eval()

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

def extract_attended_sentence_modernbert(premise_str, input_ids_1d, attn_row, tokenizer):
    tokens = tokenizer.convert_ids_to_tokens(input_ids_1d.tolist())
    attn_np = attn_row.float().cpu().numpy()

    # ModernBERT uses [SEP] as the separator. Format: [CLS] Claim [SEP] Spec Prose [SEP]
    sep_indices = [i for i, t in enumerate(tokens) if t == "[SEP]"]
    if len(sep_indices) < 2:
        return "No specific data found."

    premise_start = sep_indices[0] + 1
    premise_end = sep_indices[1]
    premise_tokens = tokens[premise_start:premise_end]
    premise_attn = attn_np[premise_start:premise_end]

    if not premise_tokens:
        return "No specific data found."

    premise_decoded = [tokenizer.convert_tokens_to_string([t]).strip() for t in premise_tokens]

    # Split on period + space to avoid splitting decimals
    raw_sentences = re.split(r'\. ', premise_str)
    sentences = [s.strip() for s in raw_sentences if s.strip()]
    if not sentences:
        return "No specific data found."

    DATA_PREFIXES = ("series", "comparison", "ranking", "axis", "trend", "min=", "max=", "categories")
    def is_data_sentence(s):
        return any(s.lower().startswith(p) for p in DATA_PREFIXES)

    data_sentences = [s for s in sentences if is_data_sentence(s)]
    candidates = data_sentences if data_sentences else sentences

    best_sentence = candidates[0]
    best_score = -1.0

    for sentence in candidates:
        sent_lower = sentence.lower()
        token_scores = [
            attn for tok_str, attn in zip(premise_decoded, premise_attn)
            if tok_str and len(tok_str) > 1 and tok_str.lower() in sent_lower
        ]
        if token_scores:
            score = float(np.mean(token_scores))
            if score > best_score:
                best_score = score
                best_sentence = sentence

    return best_sentence.strip() + "."

def synthesise_flant5_rationale(claim, pred_label, evidence_sentence):
    verdict = "supported" if pred_label == 1 else "refuted"
    prompt = (
        f"You are a chart fact-checking assistant. "
        f"The following claim has been verified as {verdict}.\n"
        f"Claim: {claim}\n"
        f"Chart evidence: {evidence_sentence}\n"
        f"Write one concise sentence explaining what the specific chart numbers show "
        f"and why the claim is therefore {verdict}. "
        f"Cite the actual values from the evidence. "
        f"Do NOT restate or paraphrase the claim.\n"
        f"Explanation:"
    )

    inputs = tokenizer_t5(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model_t5.generate(
            **inputs,
            max_new_tokens=80,
            min_new_tokens=15,
            do_sample=False,
            num_beams=4,
            no_repeat_ngram_size=3,
            repetition_penalty=1.3,
        )
    return tokenizer_t5.decode(outputs[0], skip_special_tokens=True).strip()

print("Generating rationales over combined Test 1 + Test 2 ...")
combined_raw = test_data + test2_data
combined_ds = ConcatDataset([test_dataset, test2_dataset])

references, predictions = [], []

with torch.no_grad():
    for idx, sample in enumerate(tqdm(combined_ds, desc="Rationale generation")):
        gt_explanation = combined_raw[idx].get("explanation", "")
        if not gt_explanation:
            continue

        ids = sample["input_ids"].unsqueeze(0).to(DEVICE)
        mask = sample["attention_mask"].unsqueeze(0).to(DEVICE)

        # Trigger ModernBERT to return attention matrices
        logits, attns = model(ids, mask, output_attentions=True)
        pred_val = int((torch.sigmoid(logits) >= 0.5).long().item())

        stacked_attn = torch.stack(attns, dim=0)   # [num_layers, B, heads, seq, seq]
        mean_attn = stacked_attn.mean(dim=0).mean(dim=1)  # [B, seq, seq]
        cls_attn = attns[0].mean(dim=1)[0, 0, :] # mean over heads, first batch, CLS row

        claim_text = combined_raw[idx]["claim"]
        desc = str(combined_raw[idx].get("description", "")).strip()
        ctype = str(combined_raw[idx].get("chart_type", "unknown"))
        prose_spec = combined_raw[idx]["spec_str"] # Already stored as prose in dataset
        premise_str = f"Chart Description: {desc}. Chart Type: {ctype}. Data: {prose_spec}".strip()

        evidence_sentence = extract_attended_sentence_modernbert(
            premise_str, sample["input_ids"], cls_attn, tokenizer
        )

        rationale = synthesise_flant5_rationale(claim_text, pred_val, evidence_sentence)
        references.append(gt_explanation)
        predictions.append(rationale)

print(f"Generated {len(predictions)} rationales.")

## Metrics & ChartCheck Evaluation Matrix

In [ ]:
!pip install nltk rouge-score bert-score evaluate
!pip install git+https://github.com/google-research/bleurt.git

In [ ]:
# ── NLP Metrics ──
if len(predictions) > 0:
    print("\nCalculating BLEU and ROUGE-L ...")
    rouge_sc = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    smooth = SmoothingFunction()
    bleu_list, rouge_list = [], []

    for ref, pred in zip(references, predictions):
        ref_tok = word_tokenize(ref)
        pred_tok = word_tokenize(pred)
        bleu_list.append(sentence_bleu([ref_tok], pred_tok, smoothing_function=smooth.method1))
        rouge_list.append(rouge_sc.score(ref, pred)["rougeL"].fmeasure)

    my_bleu = np.mean(bleu_list) * 100
    my_rouge = np.mean(rouge_list) * 100
    print(f"  BLEU   : {my_bleu:.1f}")
    print(f"  ROUGE-L: {my_rouge:.1f}")

    print("Calculating BERTScore (this may take a minute) ...")
    from bert_score import score as calc_bert_score
    P, R, F1_bert = calc_bert_score(predictions, references, lang="en", verbose=False)
    my_bertscore = F1_bert.mean().item() * 100
    print(f"  BERTScore F1: {my_bertscore:.1f}")
else:
    print("\nNo ground-truth explanations found. Skipping NLP metrics.")
    my_bleu, my_rouge, my_bertscore = 0.0, 0.0, 0.0

# ── Baselines from the ChartCheck paper ──
chartcheck_baselines = {
    "Model": [
        "DePlot-DeBERTa-class",
        "DePlot-FlanT5-finetune-multi",
        "MatCha-finetune-multi",
        "GPT4V (Zero-Shot)",
    ],
    "Task":        ["C",   "M",   "M",   "M"],
    "Test1_Acc":   [75.0,  65.7,  59.4,  73.8],
    "Test1_F1":    [75.0,  65.7,  59.1,  73.5],
    "Test2_Acc":   [72.5,  65.9,  61.1,  72.0],
    "Test2_F1":    [72.5,  65.8,  60.9,  71.3],
    "Avg_Acc":     [73.8,  65.8,  60.2,  72.9],
    "BLEU":        ["-",   17.3,  17.1,  10.0],
    "ROUGE-L":     ["-",   46.3,  37.3,  32.3],
    "BERTScore":   ["-",   91.5,  67.8,  89.1],
}

# ── Our result ──
our_result = {
    "Model":      ["ChartSpec + ModernBERT (Ours)"],
    "Task":       ["Pipeline"],
    "Test1_Acc":  [round(t1_acc,  1)],
    "Test1_F1":   [round(t1_f1,   1)],
    "Test2_Acc":  [round(t2_acc,  1)],
    "Test2_F1":   [round(t2_f1,   1)],
    "Avg_Acc":    [round(avg_acc, 1)],
    "BLEU":       [round(my_bleu,       1)],
    "ROUGE-L":    [round(my_rouge,      1)],
    "BERTScore":  [round(my_bertscore,  1)],
}

df_base = pd.DataFrame(chartcheck_baselines)
df_ours = pd.DataFrame(our_result)
df_final = pd.concat([df_base, df_ours], ignore_index=True)

SEP = "=" * 115
print(f"\n{SEP}")
print("  CHARTCHECK EVALUATION MATRIX — ModernBERT Pipeline")
print(SEP)
print(df_final.to_string(index=False))
print(SEP)

best_baseline_avg = 73.8
delta = avg_acc - best_baseline_avg
direction = "above" if delta >= 0 else "below"
print(f"\n  Avg accuracy vs DePlot-DeBERTa baseline : {delta:+.1f}% ({direction})")